# Varandian Optics - Refraction Demo

## Demonstration of Varandian Snell's Law (Equation 6)

**Paper:** *Varandian Optics: A Non-Euclidean Formulation of Light Propagation*  
**Author:** Pedro Coutinho Varanda  
**DOI:** [10.5281/zenodo.18529071](https://doi.org/10.5281/zenodo.18529071)

---

## Law II: Varandian Snell's Law

From the paper (Equation 6):

$$n_1 S_{K_1}(r) \sin(\theta_1) = n_2 S_{K_2}(r) \sin(\theta_2)$$

This generalizes classical Snell's law to curved spaces!

In [ ]:
# Imports
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.append('..')

from core import (
    varandian_snell,
    critical_angle,
    index_curvature_duality_spherical,
    index_curvature_duality_hyperbolic,
    space_type_name,
    SK,
)

plt.rcParams['figure.dpi'] = 100
%matplotlib inline


## Example 1: Classical Snell's Law (Verification)

First, let's verify that Varandian Snell reduces to classical Snell in Euclidean space (K=0).

In [ ]:
# Parameters
n1 = 1.0   # Air
n2 = 1.5   # Glass
K1 = 0.0   # Euclidean
K2 = 0.0   # Euclidean
r = 1.0    # Arbitrary (doesn't matter for Euclidean)

# Range of incident angles
theta1_range = np.linspace(0, np.pi/2 - 0.01, 50)
theta2_varandian = []
theta2_classical = []

for theta1 in theta1_range:
    # Varandian
    theta2_v = varandian_snell(theta1, n1, K1, n2, K2, r)
    theta2_varandian.append(theta2_v)
    
    # Classical
    theta2_c = np.arcsin(n1 * np.sin(theta1) / n2)
    theta2_classical.append(theta2_c)

# Convert to numpy arrays for plotting
theta2_varandian = np.array(theta2_varandian)
theta2_classical = np.array(theta2_classical)

# Plot
plt.figure(figsize=(10, 6))
plt.plot(np.degrees(theta1_range), np.degrees(theta2_varandian), 
         'b-', linewidth=2.5, label='Varandian Snell (K=0)')
plt.plot(np.degrees(theta1_range), np.degrees(theta2_classical), 
         'r--', linewidth=2, label='Classical Snell', alpha=0.7)

plt.xlabel('Incident Angle θ₁ (degrees)', fontsize=12)
plt.ylabel('Refracted Angle θ₂ (degrees)', fontsize=12)
plt.title('Verification: Varandian = Classical in Euclidean Space', 
          fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Check difference
max_diff = np.max(np.abs(theta2_varandian - theta2_classical))
print(f"Maximum difference: {max_diff:.2e} radians")
print("Perfect agreement! Varandian Snell reduces to classical Snell. ✓")

## Example 2: Refraction Between Different Curvatures

Now the interesting case: refraction from spherical to hyperbolic space!

In [ ]:
# Spherical to Hyperbolic
n1 = 1.0
n2 = 1.0
K1 = 1.0   # Spherical
K2 = -1.0  # Hyperbolic
r = 0.5

theta1_range = np.linspace(0.01, np.pi/2 - 0.01, 50)
theta2_range = []

for theta1 in theta1_range:
    theta2 = varandian_snell(theta1, n1, K1, n2, K2, r)
    if theta2 is not None:
        theta2_range.append(theta2)
    else:
        theta2_range.append(np.nan)  # Total internal reflection

# Plot
plt.figure(figsize=(10, 6))
plt.plot(np.degrees(theta1_range), np.degrees(theta2_range), 
         'b-', linewidth=2.5, label=f'{space_type_name(K1)} → {space_type_name(K2)}')

# Add classical for comparison
theta2_classical_comp = [np.arcsin(n1 * np.sin(t) / n2) if n1 * np.sin(t) / n2 <= 1 else np.nan 
                         for t in theta1_range]
plt.plot(np.degrees(theta1_range), np.degrees(theta2_classical_comp), 
         'r--', linewidth=2, label='Classical (for reference)', alpha=0.6)

plt.xlabel('Incident Angle θ₁ (degrees)', fontsize=12)
plt.ylabel('Refracted Angle θ₂ (degrees)', fontsize=12)
plt.title('Refraction: Spherical Space → Hyperbolic Space', 
          fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Key Observation:")
print("Refraction behaves DIFFERENTLY than in flat space!")
print("The curvature fundamentally changes how light bends at interfaces.")

## Example 3: Critical Angle

Compute critical angles for total internal reflection in curved spaces.

In [ ]:
# Glass to air in different geometries
n1 = 1.5
n2 = 1.0
r = 1.0

print("Critical Angles for Total Internal Reflection (mixed geometries):\n")

pairs = [
    (0.0, 0.0, 'Euclidean -> Euclidean'),
    (1.0, 1.0, 'Spherical -> Spherical'),
    (-1.0, -1.0, 'Hyperbolic -> Hyperbolic'),
    (1.0, 0.0, 'Spherical -> Euclidean'),
    (0.0, 1.0, 'Euclidean -> Spherical'),
    (1.0, -1.0, 'Spherical -> Hyperbolic'),
    (-1.0, 1.0, 'Hyperbolic -> Spherical'),
]

for K1, K2, label in pairs:
    theta_c = critical_angle(n1=n1, K1=K1, n2=n2, K2=K2, r=r)
    SK1 = SK(r, K1)
    SK2 = SK(r, K2)
    ratio = (n2 * SK2) / (n1 * SK1)
    if theta_c is None:
        out = 'No critical angle (ratio >= 1)'
    else:
        out = f"{np.degrees(theta_c):.2f}°"
    print(f"{label:30s}: SK1={SK1:.4f}, SK2={SK2:.4f}, ratio={ratio:.4f} -> θc = {out}")

print("\nThe critical angle depends on geometry when K1 != K2 and SK values differ.")

## Example 4: Index-Curvature Duality (Equation 7)

These refractive index profiles in FLAT space emulate curved geometries!

In [ ]:
# Spherical-emulating profile
rho_sphere = np.linspace(0, 3, 200)
n_sphere = index_curvature_duality_spherical(rho_sphere)

# Hyperbolic-emulating profile
rho_hyp = np.linspace(0, 0.95, 200)
n_hyp = index_curvature_duality_hyperbolic(rho_hyp)

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Spherical
ax1.plot(rho_sphere, n_sphere, 'b-', linewidth=2.5)
ax1.set_xlabel('Radial coordinate ρ', fontsize=12)
ax1.set_ylabel('Refractive index n(ρ)', fontsize=12)
ax1.set_title('Spherical-Emulating Profile\n$n(\\rho) = 2/(1+\\rho^2)$', 
              fontsize=13, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.set_ylim(0, 2.2)

# Hyperbolic
ax2.plot(rho_hyp, n_hyp, 'r-', linewidth=2.5)
ax2.set_xlabel('Radial coordinate ρ', fontsize=12)
ax2.set_ylabel('Refractive index n(ρ)', fontsize=12)
ax2.set_title('Hyperbolic-Emulating Profile\n$n(\\rho) = 2/(1-\\rho^2)$', 
              fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.axvline(x=1.0, color='gray', linestyle='--', alpha=0.5, label='ρ=1 (boundary)')
ax2.legend()

plt.tight_layout()
plt.show()

print("\nIndex-Curvature Duality (Law IV):")
print("These profiles in FLAT space create the SAME optical behavior")
print("as curved spaces with constant curvature!")
print("\nLeft: Luneburg lens (spherical emulation)")
print("Right: Poincaré disk (hyperbolic emulation)")

## Summary

**Key Takeaways:**

1. **Varandian Snell's Law** extends classical refraction to curved spaces
2. **Curvature matters**: Critical angles depend on geometry
3. **Index-curvature duality**: Gradient profiles can emulate curved spaces
4. **Practical applications**: Metamaterials, gradient-index optics

---

**Paper:** Varanda, P.C. (2026). *Varandian Optics*. [doi:10.5281/zenodo.18529071](https://doi.org/10.5281/zenodo.18529071)